In [1]:
import subprocess
from pathlib import Path
import pandas as pd
import datetime

date = datetime.datetime.now().strftime("%Y_%m")
# =====================================================================
# STEP 1: AUTOMATICALLY EXECUTE ALL CLEANUP NOTEBOOKS
# =====================================================================
print("=== STEP 1: RUNNING ALL PRODUCT NOTEBOOKS ===")

# Locates all product-specific cleanup notebooks across your subfolders
notebooks_to_run = list(Path(".").rglob("clean_*_stuff.ipynb"))

if not notebooks_to_run:
    print("[WARNING] No cleanup notebooks found matching 'clean_*_stuff.ipynb'.")
else:
    for nb in notebooks_to_run:
        print(f"Executing notebook: {nb}...")
        try:
            # Runs the notebook from top to bottom and updates it in place
            subprocess.run([
                "jupyter", "nbconvert", 
                "--to", "notebook", 
                "--execute", 
                "--inplace", 
                str(nb)
            ], check=True)
            print(f" -> Success: {nb.name} finished running perfectly.")
        except subprocess.CalledProcessError as e:
            print(f" [CRITICAL ERROR] Failed while running {nb.name}: {e}")

print("-" * 60)

# =====================================================================
# STEP 2: CONSOLIDATE EXCEL RESULTS INTO MASTER FILES
# =====================================================================
print("=== STEP 2: CONSOLIDATING EXCEL OUTPUTS ===")

# Define the root path and the patterns for each category of output files
ROOT_DIR = Path(".")
categories = {
    "hierarchy": "*_hierarchy_fix.xlsx",
    "quantity": "*_quantity_fix.xlsx",
    "sts": "*_sts_fix.xlsx"
}

# Loop through each category to find and stack the data
for category, pattern in categories.items():
    all_dataframes = []
    print(f"Processing category '{category.upper()}' with pattern '{pattern}'...")
    
    # Dynamically find all matching excel files generated inside the subfolders
    matching_files = list(ROOT_DIR.rglob(pattern))
    
    for file_path in matching_files:
        try:
            # Read the current excel file
            df = pd.read_excel(file_path)
            
            # Extract the product name from the folder name (e.g., 'AQUA', 'MOSAIQ')
            product_name = file_path.parent.name.replace("Mass_update_", "")
            
            # Add a source tracking column so you can filter by product later
            df['Source_Product'] = product_name
            
            all_dataframes.append(df)
            print(f" -> Loaded: {file_path.name} from {product_name} ({len(df)} rows)")
        except Exception as e:
            print(f" [ERROR] Could not read file {file_path}: {e}")
            
    # Concatenate all dataframes into a single Master file
    if all_dataframes:
        master_df = pd.concat(all_dataframes, ignore_index=True)
        
        # Define the output file path in the project root
        output_filename = f"CONSOLIDATED_{category}_{date}.xlsx"
        master_df.to_excel(output_filename, index=False)
        
        print(f"==> SUCCESS: Created '{output_filename}' with {len(master_df)} total rows.\n")
    else:
        print(f" [WARNING] No files found to consolidate for category '{category}'.\n")

print("=== ALL PROCESSES COMPLETED SUCCESSFULLY ===")

=== STEP 1: RUNNING ALL PRODUCT NOTEBOOKS ===
Executing notebook: Mass_update_AQUA\clean_AQUA_stuff.ipynb...
 -> Success: clean_AQUA_stuff.ipynb finished running perfectly.
Executing notebook: Mass_update_DOSIsoft\clean_DOSIsoft_stuff.ipynb...
 -> Success: clean_DOSIsoft_stuff.ipynb finished running perfectly.
Executing notebook: Mass_update_KAIKU\clean_KAIKU_stuff.ipynb...
 -> Success: clean_KAIKU_stuff.ipynb finished running perfectly.
Executing notebook: Mass_update_MOSAIQ\clean_MOSAIQ_stuff.ipynb...
 -> Success: clean_MOSAIQ_stuff.ipynb finished running perfectly.
------------------------------------------------------------
=== STEP 2: CONSOLIDATING EXCEL OUTPUTS ===
Processing category 'HIERARCHY' with pattern '*_hierarchy_fix.xlsx'...
 -> Loaded: mass_update_AQUA_hierarchy_fix.xlsx from AQUA (6 rows)
 -> Loaded: mass_update_DOSIsoft_hierarchy_fix.xlsx from DOSIsoft (5 rows)
 -> Loaded: mass_update_kaiku_hierarchy_fix.xlsx from KAIKU (0 rows)
 -> Loaded: mass_update_MOSAIQ_hierarc

C:\Users\falve11887\AppData\Local\Temp\ipykernel_24964\2148535913.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat(all_dataframes, ignore_index=True)
C:\Users\falve11887\AppData\Local\Temp\ipykernel_24964\2148535913.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat(all_dataframes, ignore_index=True)


==> SUCCESS: Created 'CONSOLIDATED_sts_2026_08.xlsx' with 605 total rows.

=== ALL PROCESSES COMPLETED SUCCESSFULLY ===
